# PPO on PortfolioOptimizationEnv (10-ticker sandbox)

This notebook is the same 10-stock Brazilian portfolio as `FinRL_PortfolioOptimizationEnv_Demo.ipynb`, but it trains the **on-policy PPO** agent that keeps the original seven PPO steps:

1. Actor samples logits and stores `log_prob`
2. POE executes a softmax portfolio vector and returns `ln(V_t / V_{t-1})`
3. Critic estimates `V(s)`
4. GAE advantage
5. Clipped surrogate
6. Critic MSE
7. Discard the on-policy batch

Jiang `"pg"` / EIIE is **not** used here. A runnable script version lives at `examples/ppo_poe_10ticker_sandbox.py`.

**Sandbox note:** the original EIIE demo trains for 40 episodes. This notebook defaults to 3 so it is practical to run end-to-end.

## Imports

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yfinance as yf
from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import (
    PortfolioOptimizationEnv,
)
from finrl.agents.portfolio_optimization.models import DRLAgent

logging.getLogger("matplotlib.font_manager").disabled = True
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)

TOP_BRL = [
    "VALE3.SA", "PETR4.SA", "ITUB4.SA", "BBDC4.SA",
    "BBAS3.SA", "RENT3.SA", "LREN3.SA", "PRIO3.SA",
    "WEGE3.SA", "ABEV3.SA",
]
print("tickers:", len(TOP_BRL))

## Fetch the 10-ticker sandbox

Same universe and dates as the POE demo: 2011–2022, train through 2019, test 2020 / 2021 / 2022.

In [ ]:
def fetch_top_brl(start_date="2011-01-01", end_date="2022-12-31"):
    frames = []
    for tic in TOP_BRL:
        temp = yf.download(tic, start=start_date, end=end_date, auto_adjust=True, progress=True)
        if getattr(temp.columns, "nlevels", 1) != 1:
            temp.columns = temp.columns.droplevel(1)
        temp = temp.reset_index()
        temp["tic"] = tic
        frames.append(temp)
    data = pd.concat(frames, ignore_index=True).rename(
        columns={"Date": "date", "Close": "close", "High": "high", "Low": "low", "Open": "open", "Volume": "volume"}
    )
    data["date"] = pd.to_datetime(data["date"]).dt.strftime("%Y-%m-%d")
    keep = [c for c in ["date", "open", "high", "low", "close", "volume", "tic"] if c in data.columns]
    return data[keep].dropna().sort_values(["date", "tic"]).reset_index(drop=True)

portfolio_raw_df = fetch_top_brl()
portfolio_raw_df.head()

In [ ]:
portfolio_norm_df = portfolio_raw_df.copy()
for tic, group in portfolio_raw_df.groupby("tic"):
    scaler = MaxAbsScaler()
    portfolio_norm_df.loc[group.index, ["close", "high", "low"]] = scaler.fit_transform(
        group[["close", "high", "low"]]
    )
df_portfolio = portfolio_norm_df[["date", "tic", "close", "high", "low"]]

df_portfolio_train = df_portfolio[
    (df_portfolio["date"] >= "2011-01-01") & (df_portfolio["date"] < "2019-12-31")
]
df_portfolio_2020 = df_portfolio[
    (df_portfolio["date"] >= "2020-01-01") & (df_portfolio["date"] < "2020-12-31")
]
df_portfolio_2021 = df_portfolio[
    (df_portfolio["date"] >= "2021-01-01") & (df_portfolio["date"] < "2021-12-31")
]
df_portfolio_2022 = df_portfolio[
    (df_portfolio["date"] >= "2022-01-01") & (df_portfolio["date"] < "2022-12-31")
]
print(df_portfolio_train["date"].nunique(), "train dates")

## Instantiate POE and PPO

The environment is unchanged from the original demo (`initial_amount=100000`, 25 bps commission, `time_window=50`). The agent is `"ppo"` instead of `"pg"`.

In [ ]:
def make_env(df, cwd):
    Path(cwd).mkdir(parents=True, exist_ok=True)
    return PortfolioOptimizationEnv(
        df,
        initial_amount=100000,
        comission_fee_pct=0.0025,
        time_window=50,
        features=["close", "high", "low"],
        normalize_df=None,
        cwd=cwd,
    )

environment = make_env(df_portfolio_train, "results/ppo_poe_sandbox/train")
print("action space:", environment.action_space)
print("observation space:", environment.observation_space)

model_kwargs = {
    "lr": 3e-4,
    "n_steps": 128,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
    "ent_coef": 0.01,
}
policy_kwargs = {"hidden_sizes": (64, 64)}

model = DRLAgent(environment).get_model("ppo", device, model_kwargs, policy_kwargs)

### Train

Sandbox default is 3 episodes. Raise this toward 40 to match the EIIE notebook.

In [ ]:
DRLAgent.train_model(model, episodes=3)
torch.save(model.actor_critic.state_dict(), "results/ppo_poe_sandbox/policy_ppo.pt")

## Test years and uniform buy-and-hold

In [ ]:
environment_2020 = make_env(df_portfolio_2020, "results/ppo_poe_sandbox/2020")
environment_2021 = make_env(df_portfolio_2021, "results/ppo_poe_sandbox/2021")
environment_2022 = make_env(df_portfolio_2022, "results/ppo_poe_sandbox/2022")
environment_train_eval = make_env(df_portfolio_train, "results/ppo_poe_sandbox/train_eval")

PPO_results = {}
for year, env in (
    ("training", environment_train_eval),
    ("2020", environment_2020),
    ("2021", environment_2021),
    ("2022", environment_2022),
):
    DRLAgent.DRL_validation(model, env)
    PPO_results[year] = env._asset_memory["final"]

PORTFOLIO_SIZE = len(TOP_BRL)
UBAH_results = {}
for name, env in (
    ("train", make_env(df_portfolio_train, "results/ppo_poe_sandbox/ubah_train")),
    ("2020", make_env(df_portfolio_2020, "results/ppo_poe_sandbox/ubah_2020")),
    ("2021", make_env(df_portfolio_2021, "results/ppo_poe_sandbox/ubah_2021")),
    ("2022", make_env(df_portfolio_2022, "results/ppo_poe_sandbox/ubah_2022")),
):
    terminated = False
    env.reset()
    while not terminated:
        action = [0] + [1 / PORTFOLIO_SIZE] * PORTFOLIO_SIZE
        _, _, terminated, _ = env.step(action)
    UBAH_results[name] = env._asset_memory["final"]

## Plots

In [ ]:
def plot_period(title, ubah, ppo_values):
    plt.figure(figsize=(10, 5))
    plt.plot(ubah, label="Buy and Hold")
    plt.plot(ppo_values, label="PPO")
    plt.title(title)
    plt.ylabel("Portfolio value")
    plt.legend()
    plt.show()

plot_period("Performance in training period", UBAH_results["train"], PPO_results["training"])
plot_period("Performance in 2020", UBAH_results["2020"], PPO_results["2020"])
plot_period("Performance in 2021", UBAH_results["2021"], PPO_results["2021"])
plot_period("Performance in 2022", UBAH_results["2022"], PPO_results["2022"])